# IndoML T2 — Quick-win submission on RunPod A100: win8 + shift-TTA (no training)
Kaggle smoke: D win4 18.44 → D win8 + shift-TTA×2 **19.22** (+0.78 local). Picks the best of {D, B+D} × {TTA×2, TTA×4} on held-out, then builds the test ZIP.
Cell 0: paste HF + Kaggle tokens. Run top to bottom. Download `submission_tta.zip`, then TERMINATE.

In [ ]:
# 0) Setup on RunPod — paste your tokens below (do not share this notebook afterwards)
import os, sys, glob, json, subprocess, stat
os.environ['HF_TOKEN'] = 'PASTE_YOUR_HF_TOKEN_HERE'
os.environ['KAGGLE_API_TOKEN'] = 'PASTE_YOUR_KAGGLE_API_TOKEN_HERE'      # the KGAT_... token from Kaggle Settings -> API Tokens
assert os.environ['HF_TOKEN'].startswith('hf_'), 'paste your Hugging Face token'
assert os.environ['KAGGLE_API_TOKEN'].startswith('KGAT_'), 'paste your Kaggle API token'
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
tok = os.path.expanduser('~/.kaggle/access_token'); open(tok, 'w').write(os.environ['KAGGLE_API_TOKEN']); os.chmod(tok, stat.S_IRUSR | stat.S_IWUSR)
for p_ in ['/workspace/work', '/workspace/input', '/kaggle']: os.makedirs(p_, exist_ok=True)
for src, dst in [('/workspace/work', '/kaggle/working'), ('/workspace/input', '/kaggle/input')]:
    if not os.path.exists(dst): os.symlink(src, dst)          # reuse the Kaggle code paths unchanged
subprocess.run('pip -q install -U kaggle jiwer sentencepiece webrtcvad-wheels librosa einops soundfile pandas scipy matplotlib joblib datasets transformers accelerate', shell=True)
print(subprocess.run('kaggle --version', shell=True, capture_output=True, text=True).stdout)
import torch; print(torch.__version__, torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 1e9), 'GB')

In [ ]:
# 1) Data + models via the Kaggle API (explicit refs; all are datasets -> plain files)
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()
def get(refs, dest):
    os.makedirs(dest, exist_ok=True)
    for ref in refs:
        try: print('downloading', ref, flush=True); api.dataset_download_files(ref, path=dest, unzip=True, quiet=True); return
        except Exception as e: print('  failed', ref, str(e)[:100])
    raise RuntimeError(f'could not download {refs}')
get(['avin55/indoml-validation', 'avics55/indoml-validation'], '/workspace/input/datasets/avin55/indoml-validation')
get(['avin55/indoml-track2-test', 'avics55/indoml-track2-test'], '/workspace/input/datasets/avin55/indoml-track2-test')
get(['avics55/musan-noise', 'avin55/musan-noise'], '/workspace/input/musan-noise')
get(['avin55/indoml-t2-ckpt', 'avics55/indoml-t2-ckpt'], '/workspace/input/indoml-t2-ckpt')
get(['avics55/mpsenet-e10r', 'avin55/mpsenet-e10r'], '/workspace/input/mpsenet-e10r')
chk = {'validation': glob.glob('/workspace/input/**/validationMetadata.json', recursive=True),
       'test wavs (3178)': len(glob.glob('/workspace/input/datasets/avin55/indoml-track2-test/audio/*.wav')),
       'musan (930)': len(glob.glob('/workspace/input/musan-noise/**/*.wav', recursive=True)),
       'noisy asr': glob.glob('/workspace/input/**/noisy_asr_val.json', recursive=True),
       'models': sorted(os.path.basename(x) for x in glob.glob('/workspace/input/mpsenet-e10r/**/*.pt', recursive=True))}
for k, v in chk.items(): print(f'{k:18s}:', v if v else 'MISSING')

In [ ]:
!rm -rf /kaggle/working/ner && git clone -q --depth 1 https://github.com/nagshubhadip/noise-event-detection-and-removal /kaggle/working/ner
!pip -q install jiwer sentencepiece webrtcvad-wheels librosa 2>&1 | tail -1
!wget -q https://github.com/karolpiczak/ESC-50/archive/refs/heads/master.zip -O /kaggle/working/esc50.zip && unzip -q -o /kaggle/working/esc50.zip -d /kaggle/working/ && ls /kaggle/working/ESC-50-master | head
# ---- MP-SENet (MIT) ----
!rm -rf /kaggle/working/mpsenet && git clone -q --depth 1 https://github.com/yxlu-0102/MP-SENet /kaggle/working/mpsenet
!pip -q install einops 2>&1 | tail -1

In [ ]:
import sys, os, glob, json, time, random, numpy as np, torch, soundfile as sf, librosa
from pathlib import Path
VAL = Path('/kaggle/input/datasets/avin55/indoml-validation/validation')
M = Path('/kaggle/working/ner/track2_removal/versions/v-1_indoml2026_track2/modular')
d = (M/'data.py').read_text()
d = d.replace('Path("/kaggle/input/indoml-validation"),', 'Path("/kaggle/input/datasets/avin55/indoml-validation/validation"), Path("/kaggle/input/indoml-validation"),')
d = d.replace('cid = (r.get("clip_id") or r.get("id")', 'cid = (r.get("segmentFileName") or r.get("clip_id") or r.get("id")')
(M/'data.py').write_text(d); sys.path.insert(0, str(M))
HF = os.environ["HF_TOKEN"]
import config, data, evaluate
import model as model_mod
from train import train_on_sim
from inference import enhance, Transcriber
from jiwer import wer as jwer
config.ensure_dirs()
val_dir, items = data.resolve_validation()
meta = {m['segmentFileName'][:-4]: m for m in json.load(open(VAL/'validationMetadata.json', encoding='utf-8'))}
print('validation clips', len(items))
import torch.nn.functional as F

In [ ]:
import types, sys
sys.modules.setdefault('pesq', types.SimpleNamespace(pesq=lambda *a, **k: 0.0))   # PESQ not needed (no metric discriminator)
sys.path.insert(0, '/kaggle/working/mpsenet')
from env import AttrDict
from dataset import mag_pha_stft, mag_pha_istft
from models.model import MPNet, phase_losses
h = AttrDict(json.load(open('/kaggle/working/mpsenet/best_ckpt/config.json')))
dev = torch.device('cuda')
def load_mp(path='/kaggle/working/mpsenet/best_ckpt/g_best_dns'):
    m = MPNet(h).to(dev); m.load_state_dict(torch.load(path, map_location=dev, weights_only=False)['generator']); return m.eval()
def mp_forward(m, wav):                       # wav [B, T] (RMS-normalised) -> enhanced [B, T]
    amp, pha, _ = mag_pha_stft(wav, h.n_fft, h.hop_size, h.win_size, h.compress_factor)
    amp_g, pha_g, com_g = m(amp, pha)
    return mag_pha_istft(amp_g, pha_g, h.n_fft, h.hop_size, h.win_size, h.compress_factor), (amp_g, pha_g, com_g)
@torch.no_grad()
def enhance_mp(m, x, win=4.0, hop=3.0):
    x = np.asarray(x, np.float32); n = len(x); sr = 16000
    nf = float(np.sqrt(n / (np.sum(x**2) + 1e-9))); xn = torch.from_numpy(x * nf).to(dev)
    W, H = int(win * sr), int(hop * sr)
    if n <= W:
        y = mp_forward(m, xn[None])[0][0][:n]
    else:
        y = torch.zeros(n, device=dev); wsum = torch.zeros(n, device=dev)
        fade = torch.ones(W, device=dev); r = W - H; fade[:r] = torch.linspace(0, 1, r, device=dev); fade[-r:] = torch.linspace(1, 0, r, device=dev)
        s = 0
        while True:
            e = min(s + W, n); seg = xn[s:e]
            if len(seg) < W: seg = torch.nn.functional.pad(seg, (0, W - len(seg)))
            out = mp_forward(m, seg[None])[0][0][:W]
            L = e - s; f = fade[:L].clone()
            if s == 0: f[:r] = 1
            if e == n: f[-min(r, L):] = 1
            y[s:e] += out[:L] * f; wsum[s:e] += f
            if e == n: break
            s += H
        y = y / wsum.clamp_min(1e-6)
    y = (y / nf).cpu().numpy()
    return np.pad(y, (0, max(0, n - len(y))))[:n].astype(np.float32)
mp = load_mp(); print('MP-SENet params (M):', round(sum(p.numel() for p in mp.parameters()) / 1e6, 2))

In [ ]:
# Isolated validation noise events + fingerprint clustering
SR = 16000
NV = Path('/kaggle/working/bank/val_noise'); NV.mkdir(parents=True, exist_ok=True)
segs, fps, owner = [], [], []
for v in items:
    if not v['syn']: continue
    y = data.read_audio(v['path']); c = data.read_audio(v['ref']); L = min(len(y), len(c)); n = (y[:L] - c[:L]).astype(np.float32)
    for k, e in enumerate(meta.get(v['id'], {}).get('NoiseSubCategoryTimeStamp', [])):
        a, b = max(0, int((float(e['start']) - 0.02) * SR)), min(L, int((float(e['end']) + 0.02) * SR))
        if b - a < int(0.3 * SR) or np.abs(n[a:b]).max() < 1e-4: continue
        seg = n[a:b]; p = NV / f"{v['id'][-40:]}_{k}.wav"; sf.write(p, seg, SR)
        mel = librosa.power_to_db(librosa.feature.melspectrogram(y=seg, sr=SR, n_mels=48, n_fft=512, hop_length=160) + 1e-10)
        f = librosa.util.fix_length(mel, size=48, axis=1).ravel(); f = (f - f.mean()) / (f.std() + 1e-6)
        segs.append({'path': str(p), 'dur': (b - a) / SR}); fps.append(f / np.linalg.norm(f)); owner.append(v['id'])
F = np.stack(fps); cents, lab = [], np.zeros(len(F), int)
for i, f in enumerate(F):
    if cents:
        s = np.stack(cents) @ f; j = int(s.argmax())
        if s[j] > 0.95: lab[i] = j; continue
    cents.append(f); lab[i] = len(cents) - 1
ncl = len(cents); rng = np.random.default_rng(7)
print('noise events', len(F), '| clusters', ncl)
for frac in [0.35, 0.45, 0.55]:
    held = set(rng.permutation(ncl)[:int(frac * ncl)].tolist())
    clip_cl = {}
    for o, l in zip(owner, lab): clip_cl.setdefault(o, set()).add(int(l))
    HELD_CLIPS = [o for o, s in clip_cl.items() if s <= held]
    if len(HELD_CLIPS) >= 80: break
train_noise = [s for s, l in zip(segs, lab) if int(l) not in held]
print(f'held-out clusters {len(held)}/{ncl} | held-out eval clips {len(HELD_CLIPS)} | train noise segments {len(train_noise)}')

In [ ]:
# Load models D and B from the mpsenet-e10r dataset
import gc, zipfile, torch.nn.functional as F
def find_ckpt(key):
    plain = [x for x in glob.glob(f'/kaggle/input/**/*{key}*.pt', recursive=True) if os.path.isfile(x)]
    assert plain, f'{key} not found'; return plain[0]
D_CK, B_CK = find_ckpt('anneal'), find_ckpt('warm_e8')
mD, mB = load_mp(D_CK), load_mp(B_CK); print('D:', D_CK, '| B:', B_CK)
del mp; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# Held-out selection (228 synthetic + 400 natural): {D, B+D} x {win8+TTA2, win8+TTA4}
from inference import check_output, Transcriber
NOISY = glob.glob('/kaggle/input/**/noisy_asr_val.json', recursive=True)[0]
noisy = json.load(open(NOISY)); asr = Transcriber(HF)
byid = {v['id']: v for v in items}; NAT = [v['id'] for v in items if not v['syn']]
rng = np.random.RandomState(0); NS = [NAT[i] for i in rng.choice(len(NAT), 400, replace=False)]
H = [i for i in HELD_CLIPS if i in byid]; ids = H + NS
X = {i: data.read_audio(byid[i]['path']) for i in ids}; REF = {i: data.read_audio(byid[i]['ref']) for i in H}; GT = {i: evaluate.normalize(byid[i]['gt']) for i in ids}
def pooled(s_, hyp): s_ = [i for i in s_ if GT[i]]; return jwer([GT[i] for i in s_], [evaluate.normalize(hyp.get(i, '')) or '@' for i in s_])
def post(x, y):
    ri, ro = np.sqrt((x**2).mean()), np.sqrt((y**2).mean()); z = y * (ri / ro) if ro > 1e-8 else y
    pk = np.abs(z).max(); return (z * (0.99 / pk) if pk > 0.99 else z).astype(np.float32)
def enhance_tta(mm, x, shifts=(0,), win=4.0, hop=3.0):
    return np.mean([enhance_mp(mm, np.pad(x, (s, 0)), win=win, hop=hop)[s:s + len(x)] for s in shifts], 0).astype(np.float32)
MODELS = {'B': mB, 'D': mD}
def score(parts, **kw):
    out = {i: post(X[i], np.mean([enhance_tta(MODELS[q], X[i], **kw) for q in parts], 0)) for i in ids}
    si = float(np.mean([evaluate.si_sdr(REF[i], out[i]) for i in H])); hyp = {}
    for k in range(0, len(ids), 32):
        for i, t in zip(ids[k:k+32], asr.transcribe_many([out[i] for i in ids[k:k+32]])): hyp[i] = t
    ds, dn = 100 * (pooled(H, noisy) - pooled(H, hyp)), 100 * (pooled(NS, noisy) - pooled(NS, hyp))
    return si + 0.32 * ds + 0.68 * dn, dict(si_sdr=round(si, 2), dWER_syn=round(ds, 2), dWER_nat400=round(dn, 2))
T2 = dict(shifts=(0, 50), win=8.0, hop=6.0); T4 = dict(shifts=(0, 25, 50, 75), win=8.0, hop=6.0)
RES = {}
for name, parts, kw in [('D win8+TTA2', ['D'], T2), ('D win8+TTA4', ['D'], T4), ('B+D win8+TTA2', ['B', 'D'], T2), ('B+D win8+TTA4', ['B', 'D'], T4)]:
    c, info = score(parts, **kw); RES[name] = (c, parts, kw); print(name, info, '| combined', round(c, 2), flush=True)
BEST = max(RES, key=lambda k: RES[k][0]); BEST_PARTS, BEST_KW = RES[BEST][1], RES[BEST][2]
print('CHOSEN:', BEST, round(RES[BEST][0], 2), '| reference: B+D win4 = 18.55 -> LB 14.81; D win8+TTA2 on Kaggle = 19.22')
print('test passes needed:', len(BEST_PARTS) * len(BEST_KW['shifts']), '(~13 min each on A100)')

In [ ]:
import soundfile as sf, zipfile, time
from collections import Counter
from pathlib import Path
test_files = sorted(Path('/kaggle/input/datasets/avin55/indoml-track2-test/audio').glob('*.wav')); print('test clips', len(test_files))
sub = Path('/workspace/work/t2_submission'); sub.mkdir(exist_ok=True); fails = Counter(); t0 = time.time(); parts = BEST_PARTS
for k, p in enumerate(test_files):
    w = data.read_audio(p); y = post(w, np.mean([enhance_tta(MODELS[q], w, **BEST_KW) for q in parts], 0)); c = check_output(w, y)
    for key in ['finite', 'len_match', 'mono', 'not_silent', 'level_ok', 'no_clip']:
        if not c[key]: fails[key] += 1
    assert c['finite'] and c['len_match'] and c['mono'], p.name
    sf.write(sub / p.name, y, 16000, subtype='PCM_16')
    if k % 500 == 0: print(k, f'{(time.time()-t0)/60:.1f} min', flush=True)
print('enhanced in', round((time.time()-t0)/60, 1), 'min | fails', dict(fails), '| combo', BEST)
names = [p.name for p in test_files]; texts = {}
for i in range(0, len(names), 32):
    for n, t in zip(names[i:i+32], asr.transcribe_paths([str(sub / n) for n in names[i:i+32]])): texts[n[:-4]] = (t or '').strip()
with open('/workspace/work/transcripts.jsonl', 'w', encoding='utf-8') as f:
    for n in names: f.write(json.dumps({'clip_id': n[:-4], 'text': texts.get(n[:-4], '')}, ensure_ascii=False) + '\n')
zp = '/workspace/work/submission_tta.zip'
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write('/workspace/work/transcripts.jsonl', 'transcripts.jsonl')
    for n in names: z.write(sub / n, n)
nl = zipfile.ZipFile(zp).namelist(); print('ZIP entries', len(nl), '| wavs', sum(x.endswith('.wav') for x in nl), '| empty transcripts', sum(1 for v in texts.values() if not v), '| MB', round(os.path.getsize(zp)/1e6))
print('>>> Download /workspace/work/submission_tta.zip, THEN stop/terminate the pod <<<')